In [1]:
from pathlib import Path
import json
import os
from collections import Counter, defaultdict

BASE = Path("/home/tahiti/MARINE_DATASETS")
FATHOM = BASE / "FathomNet"

print("=" * 80)
print("BASE:", BASE)
print("BASE exists:", BASE.exists())
print("FATHOM:", FATHOM)
print("FATHOM exists:", FATHOM.exists())
print("=" * 80)

# ------------------------------------------------------------
# 1. Top-level structure
# ------------------------------------------------------------
print("\n[1] TOP-LEVEL FILES/FOLDERS")
if BASE.exists():
    for p in sorted(BASE.iterdir()):
        try:
            size = ""
            if p.is_file():
                size = f"{p.stat().st_size / 1024:.1f} KB"
            print(f"{'DIR ' if p.is_dir() else 'FILE'}  {p.name:40s} {size}")
        except Exception as e:
            print("ERR ", p, e)
else:
    print("BASE folder does not exist")

# ------------------------------------------------------------
# 2. Important scripts
# ------------------------------------------------------------
print("\n[2] PYTHON SCRIPTS")
scripts = sorted(BASE.glob("*.py"))
if scripts:
    for s in scripts:
        print(f"- {s.name} | {s.stat().st_size / 1024:.1f} KB")
        try:
            head = s.read_text(errors="ignore").splitlines()[:5]
            print("  first lines:")
            for line in head:
                print("   ", line[:120])
        except Exception as e:
            print("  cannot read:", e)
else:
    print("No .py scripts found in BASE")

# ------------------------------------------------------------
# 3. FathomNet directory structure
# ------------------------------------------------------------
print("\n[3] FATHOMNET STRUCTURE")
if FATHOM.exists():
    for p in sorted(FATHOM.iterdir()):
        print(f"{'DIR ' if p.is_dir() else 'FILE'}  {p.name}")
else:
    print("FathomNet folder does not exist")

# ------------------------------------------------------------
# 4. Count images
# ------------------------------------------------------------
print("\n[4] IMAGE COUNTS")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

img_root = FATHOM / "images"
all_images = []

if img_root.exists():
    for ext in image_exts:
        all_images.extend(img_root.rglob(f"*{ext}"))
        all_images.extend(img_root.rglob(f"*{ext.upper()}"))

    all_images = sorted(set(all_images))
    print("Image root:", img_root)
    print("Total images:", len(all_images))

    by_ext = Counter(p.suffix.lower() for p in all_images)
    print("By extension:", dict(by_ext))

    by_folder = Counter(p.parent.name for p in all_images)
    print("\nTop folders/classes:")
    for cls, n in by_folder.most_common(30):
        print(f"{cls:45s} {n}")

    print("\nFirst 20 images:")
    for p in all_images[:20]:
        print(" -", p)
else:
    print("No images folder:", img_root)

# ------------------------------------------------------------
# 5. Metadata check
# ------------------------------------------------------------
print("\n[5] METADATA CHECK")
meta_candidates = [
    FATHOM / "metadata" / "downloaded_records.json",
    FATHOM / "annotations" / "all_records.json",
    FATHOM / "metadata" / "all_records.json",
]

for mp in meta_candidates:
    print("\nMetadata candidate:", mp)
    print("Exists:", mp.exists())
    if mp.exists():
        print("Size:", f"{mp.stat().st_size / 1024:.1f} KB")
        try:
            data = json.loads(mp.read_text())
            print("Records:", len(data))

            if len(data) > 0:
                print("First record keys:", list(data[0].keys()))
                print("First record:")
                print(json.dumps(data[0], indent=2)[:1500])

                concepts = Counter(str(r.get("concept_query", r.get("concept", "UNKNOWN"))) for r in data)
                print("\nTop concepts in metadata:")
                for c, n in concepts.most_common(30):
                    print(f"{c:45s} {n}")

                downloaded = Counter(str(r.get("downloaded", "NA")) for r in data)
                if downloaded:
                    print("\nDownloaded flags:", dict(downloaded))

                existing_paths = 0
                missing_paths = 0
                for r in data:
                    ip = r.get("image_path") or r.get("local_path")
                    if ip:
                        if Path(ip).exists():
                            existing_paths += 1
                        else:
                            missing_paths += 1
                print("Metadata image paths existing:", existing_paths)
                print("Metadata image paths missing:", missing_paths)

        except Exception as e:
            print("Could not parse metadata:", repr(e))

# ------------------------------------------------------------
# 6. VOC annotations / XML check
# ------------------------------------------------------------
print("\n[6] ANNOTATION FILES")
ann_roots = [
    FATHOM / "voc_annotations",
    FATHOM / "annotations",
    FATHOM / "labels",
]

for ar in ann_roots:
    print("\nAnnotation root:", ar)
    print("Exists:", ar.exists())
    if ar.exists():
        xmls = list(ar.rglob("*.xml"))
        txts = list(ar.rglob("*.txt"))
        jsons = list(ar.rglob("*.json"))
        print("XML:", len(xmls))
        print("TXT:", len(txts))
        print("JSON:", len(jsons))
        for p in (xmls[:5] + txts[:5] + jsons[:5]):
            print(" -", p)

# ------------------------------------------------------------
# 7. Disk usage
# ------------------------------------------------------------
print("\n[7] DISK USAGE")
def human_size(num):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num < 1024:
            return f"{num:.2f} {unit}"
        num /= 1024
    return f"{num:.2f} PB"

def folder_size(path):
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except Exception:
            pass
    return total

print("BASE size:", human_size(folder_size(BASE)))
print("FATHOM size:", human_size(folder_size(FATHOM)))
print("Images size:", human_size(folder_size(img_root)))

# ------------------------------------------------------------
# 8. Final diagnosis
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("DIAGNOSIS")
print("=" * 80)

if not BASE.exists():
    print("BAD: /home/tahiti/MARINE_DATASETS does not exist")
elif not FATHOM.exists():
    print("BAD: FathomNet folder does not exist")
elif not img_root.exists():
    print("BAD: FathomNet/images does not exist. Downloader probably did not create images.")
elif len(all_images) == 0:
    print("BAD: images folder exists, but contains 0 images.")
else:
    print("OK: images found:", len(all_images))

if scripts:
    print("OK: scripts found:", [s.name for s in scripts])
else:
    print("WARN: no scripts found in /home/tahiti/MARINE_DATASETS")

BASE: /home/tahiti/MARINE_DATASETS
BASE exists: True
FATHOM: /home/tahiti/MARINE_DATASETS/FathomNet
FATHOM exists: True

[1] TOP-LEVEL FILES/FOLDERS
DIR   .ipynb_checkpoints                       
DIR   FathomNet                                
FILE  Untitled.ipynb                           0.1 KB
FILE  download_fathomnet_real.py               3.0 KB

[2] PYTHON SCRIPTS
- download_fathomnet_real.py | 3.0 KB
  first lines:
    import json
    import os
    import time
    from pathlib import Path
    from urllib.request import urlretrieve

[3] FATHOMNET STRUCTURE
DIR   images
DIR   metadata

[4] IMAGE COUNTS
Image root: /home/tahiti/MARINE_DATASETS/FathomNet/images
Total images: 1059
By extension: {'.png': 889, '.jpg': 170}

Top folders/classes:
Asthenactis_fisheri                           100
Coralliidae                                   100
Corallimorphus_pilatus                        100
black_coral                                   100
bony_fish                                    

In [2]:
# ============================================================
# DOWNLOAD / CACHE OPEN-VOCABULARY FOUNDATION MODELS
# for MarineOV baselines
# Run this in Jupyter
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path("/home/tahiti/MARINE_DATASETS")
MODEL_DIR = BASE / "MARINE_MODELS"
HF_HOME = MODEL_DIR / "hf_cache"
ULTRA_HOME = MODEL_DIR / "ultralytics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
ULTRA_HOME.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME)
os.environ["TORCH_HOME"] = str(MODEL_DIR / "torch_cache")
os.environ["ULTRALYTICS_SETTINGS_DIR"] = str(ULTRA_HOME)

print("MODEL_DIR:", MODEL_DIR)
print("HF_HOME:", HF_HOME)

# ------------------------------------------------------------
# 1. Install packages
# ------------------------------------------------------------
packages = [
    "torch",
    "torchvision",
    "transformers>=4.40.0",
    "huggingface_hub",
    "accelerate",
    "safetensors",
    "pillow",
    "opencv-python",
    "ultralytics",
    "open_clip_torch",
]

print("\n[1] Installing packages...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", *packages
])

# ------------------------------------------------------------
# 2. Download HuggingFace models
# ------------------------------------------------------------
from huggingface_hub import snapshot_download

hf_models = {
    "grounding_dino_tiny": "IDEA-Research/grounding-dino-tiny",
    "grounding_dino_base": "IDEA-Research/grounding-dino-base",
    "owlvit_base": "google/owlvit-base-patch32",
    "owlvit_large": "google/owlvit-large-patch14",
    "clip_vit_base": "openai/clip-vit-base-patch32",
    "clip_vit_large": "openai/clip-vit-large-patch14",
}

print("\n[2] Downloading HuggingFace models...")

downloaded_hf = {}

for name, repo_id in hf_models.items():
    print(f"\n--- {name}: {repo_id} ---")
    try:
        local_path = snapshot_download(
            repo_id=repo_id,
            cache_dir=str(HF_HOME),
            local_dir=str(MODEL_DIR / name),
            local_dir_use_symlinks=False,
            resume_download=True,
        )
        downloaded_hf[name] = local_path
        print("OK:", local_path)
    except Exception as e:
        downloaded_hf[name] = None
        print("FAILED:", repo_id)
        print("ERROR:", repr(e))

# ------------------------------------------------------------
# 3. Download YOLO-World models
# ------------------------------------------------------------
print("\n[3] Downloading YOLO-World models...")

from ultralytics import YOLO

yolo_world_models = [
    "yolov8s-worldv2.pt",
    "yolov8m-worldv2.pt",
]

downloaded_yolo = {}

for model_name in yolo_world_models:
    print(f"\n--- {model_name} ---")
    try:
        model = YOLO(model_name)
        downloaded_yolo[model_name] = "loaded_by_ultralytics"
        print("OK:", model_name)
    except Exception as e:
        downloaded_yolo[model_name] = None
        print("FAILED:", model_name)
        print("ERROR:", repr(e))

# ------------------------------------------------------------
# 4. Download OpenCLIP model
# ------------------------------------------------------------
print("\n[4] Downloading OpenCLIP model...")

downloaded_openclip = {}

try:
    import open_clip
    import torch

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32",
        pretrained="laion2b_s34b_b79k",
        cache_dir=str(MODEL_DIR / "openclip_cache"),
    )
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    downloaded_openclip["ViT-B-32/laion2b_s34b_b79k"] = "OK"
    print("OK: OpenCLIP ViT-B-32 laion2b_s34b_b79k")
except Exception as e:
    downloaded_openclip["ViT-B-32/laion2b_s34b_b79k"] = None
    print("FAILED: OpenCLIP")
    print("ERROR:", repr(e))

# ------------------------------------------------------------
# 5. Quick load test for HF models
# ------------------------------------------------------------
print("\n[5] Quick load test...")

load_tests = {}

try:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

    repo = "IDEA-Research/grounding-dino-tiny"
    proc = AutoProcessor.from_pretrained(repo, cache_dir=str(HF_HOME))
    mod = AutoModelForZeroShotObjectDetection.from_pretrained(repo, cache_dir=str(HF_HOME))
    load_tests["grounding_dino_tiny"] = "OK"
    print("OK: GroundingDINO tiny")
except Exception as e:
    load_tests["grounding_dino_tiny"] = repr(e)
    print("FAILED: GroundingDINO tiny", repr(e))

try:
    from transformers import OwlViTProcessor, OwlViTForObjectDetection

    repo = "google/owlvit-base-patch32"
    proc = OwlViTProcessor.from_pretrained(repo, cache_dir=str(HF_HOME))
    mod = OwlViTForObjectDetection.from_pretrained(repo, cache_dir=str(HF_HOME))
    load_tests["owlvit_base"] = "OK"
    print("OK: OWL-ViT base")
except Exception as e:
    load_tests["owlvit_base"] = repr(e)
    print("FAILED: OWL-ViT base", repr(e))

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print("\nHuggingFace models:")
for k, v in downloaded_hf.items():
    print(f"{k:25s} -> {v}")

print("\nYOLO-World models:")
for k, v in downloaded_yolo.items():
    print(f"{k:25s} -> {v}")

print("\nOpenCLIP:")
for k, v in downloaded_openclip.items():
    print(f"{k:35s} -> {v}")

print("\nLoad tests:")
for k, v in load_tests.items():
    print(f"{k:25s} -> {v}")

print("\nDisk usage:")
subprocess.call(["du", "-sh", str(MODEL_DIR)])
print("\nModel folder:", MODEL_DIR)

MODEL_DIR: /home/tahiti/MARINE_DATASETS/MARINE_MODELS
HF_HOME: /home/tahiti/MARINE_DATASETS/MARINE_MODELS/hf_cache

[1] Installing packages...
  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached triton-3.7.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)


  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl (532.3 MB)
Using cached triton-3.7.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (201.5 MB)
Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.6 MB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.6 MB/s  0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 6.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 2.8 MB/s  0:00:20 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 1.9 MB/s  0:00:03 eta 0:00:010:01:01
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully un

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.12.0 which is incompatible.
/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[2] Downloading HuggingFace models...

--- grounding_dino_tiny: IDEA-Research/grounding-dino-tiny ---


/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 11 files: 100%|███████████████████████| 11/11 [00:53<00:00,  4.83s/it]


...

Fetching 12 files: 100%|███████████████████████| 12/12 [00:51<00:00,  4.29s/it]


OK: /home/tahiti/MARINE_DATASETS/MARINE_MODELS/clip_vit_base

--- clip_vit_large: openai/clip-vit-large-patch14 ---


Fetching 13 files: 100%|███████████████████████| 13/13 [02:17<00:00, 10.60s/it]


OK: /home/tahiti/MARINE_DATASETS/MARINE_MODELS/clip_vit_large

[3] Downloading YOLO-World models...


RuntimeError: NumPy was built with baseline optimizations: 
(X86_V2) but your machine doesn't support:
(X86_V2).

In [1]:
import sys, subprocess, os

cmds = [
    [sys.executable, "-m", "pip", "uninstall", "-y", "numpy", "opencv-python", "opencv-python-headless", "opencv-contrib-python"],
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"],
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "opencv-python-headless==4.9.0.80"],
    [sys.executable, "-m", "pip", "install", "-U", "ultralytics"],
]

for c in cmds:
    print("\nRUN:", " ".join(c))
    subprocess.check_call(c)

print("\nDONE. Now restart Jupyter kernel, then run the next cell.")


RUN: /home/tahiti/Malashin_Projects/.venv_a100/bin/python -m pip uninstall -y numpy opencv-python opencv-python-headless opencv-contrib-python



RUN: /home/tahiti/Malashin_Projects/.venv_a100/bin/python -m pip install --no-cache-dir --force-reinstall numpy==1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 868.5 kB/s  0:00:20 eta 0:00:010:00:02


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.4.64 requires opencv-python>=4.6.0, which is not installed.



RUN: /home/tahiti/Malashin_Projects/.venv_a100/bin/python -m pip install --no-cache-dir --force-reinstall opencv-python-headless==4.9.0.80
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 1.8 MB/s  0:00:27 eta 0:00:010:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 3.1 MB/s  0:00:05a 0:00:01m eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [opencv-python-headless]2 [opencv-python-headless]

RUN: /home/tahiti/Malashin_Projects/.venv_a100/bin/python -m pip install -U ultralytics


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.4.64 requires opencv-python>=4.6.0, which is not installed.
scipy 1.12.0 requires numpy<1.29.0,>=1.22.4, but you have numpy 2.4.6 which is incompatible.


  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.1 MB/s  0:00:00
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.12.0
    Uninstalling scipy-1.12.0:
      Successfully uninstalled scipy-1.12.0
  Attempting uninstall: ultralytics╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [opencv-python]
    Found existing installation: ultralytics 8.4.64━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [opencv-python]
    Uninstalling ultralytics-8.4.64:38;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3

In [4]:
import numpy as np
import cv2

print(np.__version__)
print(cv2.__version__)

2.4.6
4.9.0


In [6]:
# ============================================================
# CHECK MARINE_MODELS: files + load tests + minimal inference
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import traceback

BASE = Path("/home/tahiti/MARINE_DATASETS")
MODEL_DIR = BASE / "MARINE_MODELS"
FATHOM_DIR = BASE / "FathomNet"
IMG_DIR = FATHOM_DIR / "images"

HF_HOME = MODEL_DIR / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME)
os.environ["TORCH_HOME"] = str(MODEL_DIR / "torch_cache")
os.environ["ULTRALYTICS_SETTINGS_DIR"] = str(MODEL_DIR / "ultralytics")

print("=" * 90)
print("PYTHON:", sys.executable)
print("BASE:", BASE, "| exists:", BASE.exists())
print("MODEL_DIR:", MODEL_DIR, "| exists:", MODEL_DIR.exists())
print("HF_HOME:", HF_HOME, "| exists:", HF_HOME.exists())
print("=" * 90)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def human_size(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def folder_size(path):
    total = 0
    if not Path(path).exists():
        return 0
    for p in Path(path).rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def status(name, ok, msg=""):
    mark = "OK" if ok else "FAIL"
    print(f"{mark:5s} | {name:30s} | {msg}")

# ------------------------------------------------------------
# 1. Basic imports
# ------------------------------------------------------------

print("\n[1] BASIC IMPORT CHECK")

try:
    import numpy as np
    status("numpy", True, np.__version__)
except Exception as e:
    status("numpy", False, repr(e))

try:
    import torch
    status("torch", True, f"{torch.__version__} | cuda={torch.cuda.is_available()}")
except Exception as e:
    status("torch", False, repr(e))

try:
    from PIL import Image
    status("PIL", True, "OK")
except Exception as e:
    status("PIL", False, repr(e))

try:
    import cv2
    status("cv2", True, cv2.__version__)
except Exception as e:
    status("cv2", False, repr(e))

try:
    import transformers
    status("transformers", True, transformers.__version__)
except Exception as e:
    status("transformers", False, repr(e))

try:
    import ultralytics
    status("ultralytics", True, ultralytics.__version__)
except Exception as e:
    status("ultralytics", False, repr(e))

# ------------------------------------------------------------
# 2. Folder structure and sizes
# ------------------------------------------------------------

print("\n[2] MODEL FOLDER STRUCTURE")

if MODEL_DIR.exists():
    for p in sorted(MODEL_DIR.iterdir()):
        try:
            if p.is_dir():
                print(f"DIR  {p.name:35s} {human_size(folder_size(p))}")
            else:
                print(f"FILE {p.name:35s} {human_size(p.stat().st_size)}")
        except Exception as e:
            print("ERR ", p, e)
else:
    print("MODEL_DIR does not exist")

print("\nTotal MARINE_MODELS size:", human_size(folder_size(MODEL_DIR)))

# ------------------------------------------------------------
# 3. Expected HF model dirs
# ------------------------------------------------------------

print("\n[3] EXPECTED HUGGINGFACE MODEL DIRS")

expected_dirs = {
    "grounding_dino_tiny": MODEL_DIR / "grounding_dino_tiny",
    "grounding_dino_base": MODEL_DIR / "grounding_dino_base",
    "owlvit_base": MODEL_DIR / "owlvit_base",
    "owlvit_large": MODEL_DIR / "owlvit_large",
    "clip_vit_base": MODEL_DIR / "clip_vit_base",
    "clip_vit_large": MODEL_DIR / "clip_vit_large",
}

for name, path in expected_dirs.items():
    if path.exists():
        files = list(path.rglob("*"))
        n_files = sum(1 for x in files if x.is_file())
        status(name, True, f"{n_files} files | {human_size(folder_size(path))}")
    else:
        status(name, False, f"missing: {path}")

# ------------------------------------------------------------
# 4. YOLO-World weights search
# ------------------------------------------------------------

print("\n[4] YOLO-WORLD WEIGHTS SEARCH")

yolo_names = [
    "yolov8s-worldv2.pt",
    "yolov8m-worldv2.pt",
    "yolov8l-worldv2.pt",
    "yolov8x-worldv2.pt",
]

search_roots = [
    MODEL_DIR,
    BASE,
    Path("/home/tahiti"),
]

found_yolo = {}

for yn in yolo_names:
    found = []
    for root in search_roots:
        if root.exists():
            try:
                found.extend(root.rglob(yn))
            except Exception:
                pass
    found = sorted(set(found))
    found_yolo[yn] = found
    if found:
        for f in found:
            status(yn, True, f"{f} | {human_size(f.stat().st_size)}")
    else:
        status(yn, False, "not found")

# ------------------------------------------------------------
# 5. Find a sample image
# ------------------------------------------------------------

print("\n[5] SAMPLE IMAGE")

image_exts = [".jpg", ".jpeg", ".png", ".webp"]
sample_image = None

if IMG_DIR.exists():
    imgs = []
    for ext in image_exts:
        imgs += list(IMG_DIR.rglob(f"*{ext}"))
        imgs += list(IMG_DIR.rglob(f"*{ext.upper()}"))
    imgs = sorted(set(imgs))
    print("FathomNet images found:", len(imgs))
    if imgs:
        sample_image = imgs[0]
        print("Sample image:", sample_image)
else:
    print("No image dir:", IMG_DIR)

if sample_image is None:
    # create dummy image for model load smoke tests
    from PIL import Image
    sample_image = MODEL_DIR / "_dummy_test_image.jpg"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    Image.new("RGB", (640, 480), color=(30, 80, 120)).save(sample_image)
    print("Created dummy sample:", sample_image)

# ------------------------------------------------------------
# 6. Load and smoke-test CLIP
# ------------------------------------------------------------

print("\n[6] CLIP LOAD TEST")

try:
    import torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel

    clip_path = MODEL_DIR / "clip_vit_base"
    if not clip_path.exists():
        clip_path = "openai/clip-vit-base-patch32"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(str(clip_path)).to(device)
    processor = CLIPProcessor.from_pretrained(str(clip_path))

    image = Image.open(sample_image).convert("RGB")
    texts = ["a fish", "a crab", "a coral", "a sponge", "underwater marine animal"]

    inputs = processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        out = model(**inputs)
        probs = out.logits_per_image.softmax(dim=1)[0].detach().cpu().tolist()

    pairs = sorted(zip(texts, probs), key=lambda x: x[1], reverse=True)
    status("CLIP ViT-B/32", True, "loaded + inference OK")
    print("Top CLIP scores:")
    for t, p in pairs:
        print(f"  {t:30s} {p:.4f}")

    del model, processor

except Exception as e:
    status("CLIP ViT-B/32", False, repr(e))
    traceback.print_exc(limit=2)

# ------------------------------------------------------------
# 7. Load and smoke-test OWL-ViT
# ------------------------------------------------------------

print("\n[7] OWL-VIT LOAD TEST")

try:
    import torch
    from PIL import Image
    from transformers import OwlViTProcessor, OwlViTForObjectDetection

    owl_path = MODEL_DIR / "owlvit_base"
    if not owl_path.exists():
        owl_path = "google/owlvit-base-patch32"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = OwlViTProcessor.from_pretrained(str(owl_path))
    model = OwlViTForObjectDetection.from_pretrained(str(owl_path)).to(device)
    model.eval()

    image = Image.open(sample_image).convert("RGB")
    texts = [["a fish", "a crab", "a coral", "a sponge", "a shark"]]

    inputs = processor(text=texts, images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.Tensor([image.size[::-1]]).to(device)
    results = processor.post_process_object_detection(
        outputs=outputs,
        threshold=0.05,
        target_sizes=target_sizes
    )[0]

    n_det = len(results["scores"])
    status("OWL-ViT base", True, f"loaded + inference OK | detections={n_det}")

    for score, label, box in zip(results["scores"][:10], results["labels"][:10], results["boxes"][:10]):
        print(f"  {texts[0][label.item()]:20s} score={score.item():.3f} box={[round(x, 1) for x in box.tolist()]}")

    del model, processor

except Exception as e:
    status("OWL-ViT base", False, repr(e))
    traceback.print_exc(limit=2)

# ------------------------------------------------------------
# 8. Load and smoke-test GroundingDINO
# ------------------------------------------------------------

print("\n[8] GROUNDINGDINO LOAD TEST")

try:
    import torch
    from PIL import Image
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

    gdino_path = MODEL_DIR / "grounding_dino_base"
    if not gdino_path.exists():
        gdino_path = MODEL_DIR / "grounding_dino_tiny"
    if not gdino_path.exists():
        gdino_path = "IDEA-Research/grounding-dino-base"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(str(gdino_path))
    model = AutoModelForZeroShotObjectDetection.from_pretrained(str(gdino_path)).to(device)
    model.eval()

    image = Image.open(sample_image).convert("RGB")

    # GroundingDINO обычно любит текст с точками между классами
    text = "a fish. a crab. a coral. a sponge. a shark."

    inputs = processor(images=image, text=text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.20,
        text_threshold=0.20,
        target_sizes=[image.size[::-1]]
    )[0]

    n_det = len(results["scores"])
    status("GroundingDINO", True, f"loaded + inference OK | detections={n_det}")

    labels = results.get("labels", [""] * n_det)
    for score, label, box in zip(results["scores"][:10], labels[:10], results["boxes"][:10]):
        print(f"  {str(label):20s} score={float(score):.3f} box={[round(float(x), 1) for x in box.tolist()]}")

    del model, processor

except Exception as e:
    status("GroundingDINO", False, repr(e))
    traceback.print_exc(limit=2)

# ------------------------------------------------------------
# 9. Load and smoke-test YOLO-World
# ------------------------------------------------------------

print("\n[9] YOLO-WORLD LOAD TEST")

try:
    from ultralytics import YOLO

    yolo_path = None
    for yn in ["yolov8s-worldv2.pt", "yolov8m-worldv2.pt"]:
        if found_yolo.get(yn):
            yolo_path = found_yolo[yn][0]
            break

    if yolo_path is None:
        yolo_path = "yolov8s-worldv2.pt"  # will try download via ultralytics

    model = YOLO(str(yolo_path))

    # set text classes for open-vocab detection
    classes = ["fish", "crab", "coral", "sponge", "shark"]
    try:
        model.set_classes(classes)
    except Exception as e:
        print("WARN: set_classes failed:", repr(e))

    results = model.predict(
        source=str(sample_image),
        imgsz=640,
        conf=0.05,
        verbose=False,
        save=False
    )

    n_det = 0
    if results and hasattr(results[0], "boxes") and results[0].boxes is not None:
        n_det = len(results[0].boxes)

    status("YOLO-World", True, f"loaded + inference OK | detections={n_det}")

    if results and n_det > 0:
        boxes = results[0].boxes
        for i in range(min(10, n_det)):
            cls_id = int(boxes.cls[i].item()) if boxes.cls is not None else -1
            conf = float(boxes.conf[i].item()) if boxes.conf is not None else -1
            name = results[0].names.get(cls_id, str(cls_id)) if hasattr(results[0], "names") else str(cls_id)
            xyxy = boxes.xyxy[i].detach().cpu().tolist()
            print(f"  {name:20s} conf={conf:.3f} box={[round(float(x), 1) for x in xyxy]}")

    del model

except Exception as e:
    status("YOLO-World", False, repr(e))
    traceback.print_exc(limit=2)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL CHECK DONE")
print("=" * 90)
print("If CLIP / OWL-ViT / GroundingDINO are OK, the main zero-shot baselines are ready.")
print("If YOLO-World fails but .pt exists, it is usually an ultralytics/cv2 issue, not a missing model.")
print("MODEL_DIR:", MODEL_DIR)

PYTHON: /home/tahiti/Malashin_Projects/.venv_a100/bin/python
BASE: /home/tahiti/MARINE_DATASETS | exists: True
MODEL_DIR: /home/tahiti/MARINE_DATASETS/MARINE_MODELS | exists: True
HF_HOME: /home/tahiti/MARINE_DATASETS/MARINE_MODELS/hf_cache | exists: True

[1] BASIC IMPORT CHECK
OK    | numpy                          | 2.4.6
OK    | torch                          | 2.12.0+cpu | cuda=False
OK    | PIL                            | OK
OK    | cv2                            | 4.9.0
OK    | transformers                   | 5.11.0
OK    | ultralytics                    | 8.4.65

[2] MODEL FOLDER STRUCTURE
DIR  clip_vit_base                       1.69 GB
DIR  clip_vit_large                      6.38 GB
DIR  grounding_dino_base                 1.74 GB
DIR  grounding_dino_tiny                 1.29 GB
DIR  hf_cache                            1.75 MB
DIR  owlvit_base                         1.14 GB
DIR  owlvit_large                        1.62 GB
DIR  ultralytics                         0.00 B
DI

Traceback (most recent call last):
  File "/tmp/ipykernel_115511/1335163473.py", line 218, in <module>
    from transformers import CLIPProcessor, CLIPModel
  File "/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2262, in __getattr__
ImportError: cannot import name 'CHAT_TEMPLATE_NAME' from 'transformers.utils' (/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/__init__.py)
Traceback (most recent call last):
  File "/tmp/ipykernel_115511/1335163473.py", line 263, in <module>
    from transformers import OwlViTProcessor, OwlViTForObjectDetection
  File "/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2262, in __getattr__
ImportError: cannot import name 'CHAT_TEMPLATE_NAME' from 'transformers.utils' (/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/__init__.py)



[7] OWL-VIT LOAD TEST
FAIL  | OWL-ViT base                   | ImportError("cannot import name 'CHAT_TEMPLATE_NAME' from 'transformers.utils' (/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/__init__.py)")

[8] GROUNDINGDINO LOAD TEST
FAIL  | GroundingDINO                  | ImportError("cannot import name 'CHAT_TEMPLATE_NAME' from 'transformers.utils' (/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/__init__.py)")


Traceback (most recent call last):
  File "/tmp/ipykernel_115511/1335163473.py", line 310, in <module>
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
  File "/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2262, in __getattr__
ImportError: cannot import name 'CHAT_TEMPLATE_NAME' from 'transformers.utils' (/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/transformers/utils/__init__.py)



[9] YOLO-WORLD LOAD TEST
FAIL  | YOLO-World                     | RuntimeError('could not create a primitive')

FINAL CHECK DONE
If CLIP / OWL-ViT / GroundingDINO are OK, the main zero-shot baselines are ready.
If YOLO-World fails but .pt exists, it is usually an ultralytics/cv2 issue, not a missing model.
MODEL_DIR: /home/tahiti/MARINE_DATASETS/MARINE_MODELS


Traceback (most recent call last):
  File "/tmp/ipykernel_115511/1335163473.py", line 381, in <module>
    results = model.predict(
              ^^^^^^^^^^^^^^
  File "/home/tahiti/Malashin_Projects/.venv_a100/lib/python3.12/site-packages/ultralytics/engine/model.py", line 537, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
                                                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: could not create a primitive
